# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [ ]:
import pandas as pd
PATH = 'product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [ ]:
pop_tart = cat[cat['product_name'].str.lower().str.contains('pop')]
pop_tart
print(f'{len(pop_tart)} rows')

4 rows


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [ ]:
#strawberry
list(pop_tart['sku'])
print(f'{len(pop_tart)} rows')
#non strawberry
non_strawberry = pop_tart[~pop_tart['product_name'].str.lower().str.contains('strawberry')]
non_strawberry
print(f'{len(non_strawberry)} rows')


4 rows
0 rows


**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [ ]:
#standardizes "POPTART-STRAW" across all variations
pop_tart_item_map = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW'
}
cat['canonical_sku'] = cat['sku'].replace(pop_tart_item_map)
cat[cat['product_name'].str.lower().str.contains('pop')][['sku', 'canonical_sku','product_name']]

,sku,canonical_sku,product_name
0,PT-12,POPTART-STRAW,pop-tarts strawberry
1,SKU#459812,POPTART-STRAW,Strawberry Pop-Tart
2,POPTART-STRAW,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,POPTART-STRAW,pop-tarts strawberry


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [ ]:

category_map = {
    'Emergency Supplies':'Emergency',
    'Breakfast Foods':'Breakfast',
    'Alcoholic Beverages':'Alcohol',
    'Baby Products':'Baby',
    'Groceries':'Grocery',
}
cat['category'] = cat['category'].replace(category_map)
print(cat['category'].value_counts())

category
Emergency        8
Breakfast        3
EMERGENCY        3
ENTERTAINMENT    3
GROCERY          3
Baby             3
health           2
ELECTRONICS      2
Grocery          2
household        2
entertainment    2
ALCOHOL          1
Alcohol          1
breakfast        1
Electronics      1
grocery          1
electronics      1
Snacks           1
snacks           1
HOUSEHOLD        1
Household        1
baby             1
emergency        1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [ ]:
import re
#normalizing product names
def normalize_name(name):
    name = name.lower()
    name = re.sub(r'[^\w\s]', '', name)
    name = ' '.join(sorted(name.split()))
    return name

#applying normalization
cat['normalized_name'] = cat['product_name'].apply(normalize_name)
cat.groupby('normalized_name').filter(lambda x: len(x) > 1)

#counting different SKUs
cat['sku'] = cat['sku'].str.upper()
cat.groupby('normalized_name').filter(lambda x: len(x) > 1)

grouped = cat.groupby('normalized_name').agg(
    sku_count=('sku', 'nunique'),
    skus=('sku', lambda x: list(x.unique())),
    original_names=('product_name', lambda x: list(x.unique()))
).reset_index()

#filtering for groups containing >1 SKU
fragmented_candidates = grouped[grouped['sku_count'] > 1]

fragmented_candidates




,normalized_name,sku_count,skus,original_names
3,24pack beer light,2,"[BEER-24, LTBEER24]",[Light Beer (24-pack)]
8,4pack aa batteries,2,"[BAT-AA4, BATT_AA_4PK]",[AA Batteries 4-Pack]
12,board game monopoly,3,"[BG-MONO, MONOPOLY-01, GAME-MONO]","[Monopoly Board Game, Board Game - Monopoly]"
13,bread white,2,"[BREAD-WH, WHITEBREAD]",[WHITE BREAD]
14,butter peanut,2,"[PB-16, PNUTBTR-16]","[PEANUT BUTTER, Peanut Butter]"
15,candles emergency,2,"[CNDL-EM, CANDLE-EMRG]","[Candles (Emergency), Emergency Candles]"
16,canned soup,2,"[SOUP-CAN, CNSOUP-01]","[Soup - Canned, Canned Soup]"
17,cards playing,2,"[CARDS-01, PLAYCARD-STD]","[Playing Cards, PLAYING CARDS]"
20,chips pack variety,2,"[CHIPS-VP, CHIPVARIETY]","[Variety Pack - Chips, Chips Variety Pack]"
22,duty heavy tarp,2,"[TARP-HD, HDTARP-01]","[Heavy Duty Tarp, TARP - HEAVY DUTY]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [ ]:
# 1. Assert no null values exist in canonical_sku
assert cat['canonical_sku'].notna().all(), "Canonical SKU contains null values!"

# 2. Assert the total row count remains unchanged
assert len(cat) == len(cat['canonical_sku']), "Row count changed after consolidation!"

print("All checks passed")

All checks passed


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

One product family is 4 Pack AA batteries and I would group them by the cononical SKU "BATT_AA_4PK". Another one was white bread, and I would consolidate the 2 variations into the canonical SKU 'WHITEBREAD'.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

Data cleaning can stop becoming consolidation and becomes a business decision when seemingly similar products are combined when they actually serve different purposes. For example, a scented decorative candle might be accidentally  consolidated with an emergency candle, even though they serve different functions and shouldn't be combined.


_your answer here_